In [4]:
import numpy as np
from Utils import P_X, CanonicalUnits, GravitationalParameters
import spiceypy as spy
from scipy.integrate import nquad
from typing import Dict, Set, Optional, Union, Tuple
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
%load_ext autoreload 
%autoreload 2

### Testing Different Numerical Methods to Solve P_X integration

In [5]:
cu = CanonicalUnits()
deg = cu.deg
AU_m = cu.AU_m #m
M_sun = cu.M_sun
G = cu.G # m^3 / (kg s^2)
year = cu.year #s
mu = cu.mu
grav_params = GravitationalParameters(mu=mu)

#### Gauss–Kronrod (nquad)
(QUADPACK) uses Gauss–Kronrod adaptive rules and is very robust for localized features and integrable singularities if you provide points. Use it for the inner dimension(s) where behaviour is rough.

In [ ]:
def integrand(vz, vy, vx, z, y, x, mu):
    return P_X(x, y, z, vx, vy, vz, mu)

xmin, xmax = (0.9039710143502464, 1.0960289856497536)
ymin, ymax = (-0.09602898564975362, 0.09602898564975362)
zmin, zmax = (-0.09602898564975362, 0.09602898564975362)
vxmin, vxmax = (-0.5064245183055923, 0.5064245183055923)
vymin, vymax = (5.777598212496408, 6.790447249107593)
vzmin, vzmax = (-0.5064245183055923, 0.5064245183055923)

bounds_for_nquad = [
    (vzmin, vzmax),    # for "vz" (innermost)
    (vymin, vymax),
    (vxmin, vxmax),
    (zmin, zmax),
    (ymin, ymax),
    (xmin, xmax)       # outermost
]

opts = [
    {'epsabs': 1e-7, 'epsrel': 1e-7, 'points': [0.0]},  
    {'epsabs': 1e-4, 'epsrel': 1e-4},
    {'epsabs': 1e-4, 'epsrel': 1e-4},
    {'epsabs': 1e-4, 'epsrel': 1e-4},
    {'epsabs': 1e-4, 'epsrel': 1e-4},
    {'epsabs': 1e-4, 'epsrel': 1e-4}
]

result, info = nquad(integrand, bounds_for_nquad, args=(mu,), opts=opts)
print("integral:", result)


c:\Users\aguju\Documents\tesis\juanita\juanita\repo\TesisRepo\jacobian_calculations\Utils.py:796: RuntimeWarning: divide by zero encountered in scalar divide
  det = 1.0/np.linalg.det(J)


#### Monte Carlo Estimations

In [1]:
def montecarlo_integral(func, bounds, N=10**5, rng=None, *args, **kwargs):
    """
    Monte Carlo integral of a function func over a hyper-rectangle.

    Parameters
    ----------
    func : callable
        Function f(x, y, z, vx, vy, vz, *args, **kwargs) returning a float.
    bounds : list of (low, high)
        Integration limits for each variable.
    N : int
        Number of random samples.
    rng : int or np.random.Generator, optional
        Random seed or Generator.
    *args, **kwargs :
        Additional parameters passed to func (e.g. mu).

    Returns
    -------
    estimate : float
        Estimated value of the integral.
    err : float
        Statistical uncertainty of the estimate.
    """
    rng = np.random.default_rng(rng)
    dim = len(bounds)
    lows = np.array([b[0] for b in bounds])
    highs = np.array([b[1] for b in bounds])
    volume = np.prod(highs - lows)

    # Draw random samples uniformly
    u = rng.random((N, dim))
    samples = lows + u * (highs - lows)

    # Evaluate integrand with extra arguments
    vals = np.array([func(*s, *args, **kwargs) for s in samples])

    mean = np.mean(vals)
    std = np.std(vals, ddof=1)
    estimate = mean * volume
    err = std * volume / np.sqrt(N)

    return estimate, err


In [ ]:

xmin, xmax = (0.9039710143502464, 1.0960289856497536)
ymin, ymax = (-0.09602898564975362, 0.09602898564975362)
zmin, zmax = (-0.09602898564975362, 0.09602898564975362)
vxmin, vxmax = (-0.5064245183055923, 0.5064245183055923)
vymin, vymax = (5.777598212496408, 6.790447249107593)
vzmin, vzmax = (-0.5064245183055923, 0.5064245183055923)

N = int(1e7)
bounds = [(xmin,xmax), (ymin,ymax), (zmin,zmax), (vxmin,vxmax), (vymin,vymax), (vzmin,vzmax)]
I, dI = montecarlo_integral(P_X, bounds, N=N, mu=mu)
print(f"Integral = {I:.5e} ± {dI:.2e}")
print(f"Theoretical (integral) number of objects in volume: {I * N}")


In [ ]:
Ns = np.logspace(3, 6, 10, dtype=int)
results = [montecarlo_integral(P_X, bounds, N=N, mu=mu)[0] for n in Ns]
plt.loglog(Ns, np.abs(np.array(results) - results[-1]))
plt.xlabel('N'); plt.ylabel('error estimate')

#### Monte Carlo Importance Sampling

In [12]:
from scipy.stats import norm

def importance_mc(func, bounds, vz_peak=0, vz_sigma=0.05, N=200_000, rng=None, *args, **kwargs):
    rng = np.random.default_rng(rng)
    lows = np.array([b[0] for b in bounds])
    highs = np.array([b[1] for b in bounds])
    vol_other = np.prod((highs - lows)[:5])

    # Uniform for x,y,z,vx,vy
    u = rng.random((N, 5))
    others = lows[:5] + u * (highs[:5] - lows[:5])

    # Gaussian for vz (truncated to bounds)
    vz = rng.normal(vz_peak, vz_sigma, N)
    lz, hz = bounds[5]
    mask = (vz > lz) & (vz < hz)
    vz = vz[mask]
    others = others[:len(vz)]

    q_vz = norm.pdf(vz, vz_peak, vz_sigma)
    q_other = 1 / vol_other
    q = q_other * q_vz

    pts = np.column_stack([others, vz])
    vals = np.array([func(*p, *args, **kwargs) for p in pts])
    weights = vals / q

    estimate = np.mean(weights)
    err = np.std(weights, ddof=1) / np.sqrt(len(weights))
    return estimate, err


In [13]:
# Suppose you already have p_tildeX(x, y, z, vx, vy, vz)
N = int(1e7)
bounds = [(xmin,xmax), (ymin,ymax), (zmin,zmax), (vxmin,vxmax), (vymin,vymax), (vzmin,vzmax)]
I, dI = importance_mc(P_X, bounds, N=N, mu=mu)
print(f"Integral = {I:.5e} ± {dI:.2e}")
print(f"Theoretical (integral) number of objects in volume: {I * N}")

Integral = 6.69965e-06 ± 1.78e-07
Theoretical (integral) number of objects in volume: 66.99652431042371


In [8]:
from Utils import trasformation_X_to_E, compute_jacobian_XoE, P_E

def P_X(x: float, y: float, z: float, vx: float, vy: float, vz: float, mu: float) -> float:
    a, e, i, Omega, w, M = trasformation_X_to_E(x, y, z, vx, vy, vz, mu)
    J = compute_jacobian_XoE(a,e,i,Omega,w,M,mu)
    #det = np.linalg.det(J)
    det = 1.0/np.linalg.det(J)
    P = P_E() * abs(det)
    return P
    
def P_X_vectorized(x: np.array, y: np.array, z: np.array, vx: np.array, vy: np.array, vz: np.array, mu: float) -> np.array:
    """
    Vectorized version: x, y, vx, vy are arrays (or scalars).
    Returns array of P values.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    z = np.asarray(z)
    vx = np.asarray(vx)
    vy = np.asarray(vy)
    vz = np.asarray(vz)
    # Prepare output array
    shape = np.broadcast(x, y, z, vx, vy, vz).shape
    P = np.empty(shape, dtype=float)

    # Flatten for iteration if needed
    x_flat = x.ravel()
    y_flat = y.ravel()
    z_flat = z.ravel()
    vx_flat = vx.ravel()
    vy_flat = vy.ravel()
    vz_flat = vz.ravel()

    for idx in range(x_flat.size):
        a, e, i, Omega, w, M = trasformation_X_to_E(x_flat[idx], y_flat[idx], z_flat[idx], vx_flat[idx], vy_flat[idx], vz_flat[idx], mu)
        J = compute_jacobian_XoE(a,e,i,Omega,w,M,mu)
        det = np.linalg.det(J)
        inv_det = 1.0/det
        P.flat[idx] = P_E() * abs(inv_det)

    return P.reshape(shape)

def surface_integral_P_X(center, widths, n_points=8, mu=1):
    """
    Calculate the surface integral of P_xyvxvy in a hypercube centered at (x, y, vx, vy)
    with dimensions (dx, dy, dvx, dvy) using Gauss-Legendre quadrature.

    Parameters:
        center: tuple/list/array of (x, y, vx, vy) center
        widths: tuple/list/array of (dx, dy, dvx, dvy) side lengths
        n_points: number of quadrature points per dimension

    Returns:
        Integral (float)
    """
    from numpy.polynomial.legendre import leggauss

    x0, y0, z0, vx0, vy0, vz0 = center
    dx, dy, dz, dvx, dvy, dvz = widths

    # Get Gauss-Legendre points and weights for [-1, 1]
    pts, wts = leggauss(n_points)

    # Map points from [-1, 1] to [center-width/2, center+width/2] for each dimension
    x_pts = x0 + 0.5*dx*pts
    y_pts = y0 + 0.5*dy*pts
    z_pts = z0 + 0.5*dz*pts
    vx_pts = vx0 + 0.5*dvx*pts
    vy_pts = vy0 + 0.5*dvy*pts
    vz_pts = vz0 + 0.5*dvz*pts

    # Create meshgrid of all quadrature points
    X, Y, Z, VX, VY, VZ = np.meshgrid(x_pts, y_pts, z_pts, vx_pts, vy_pts, vz_pts, indexing='ij')
    WX, WY, WZ, WVX, WVY, WVZ = np.meshgrid(wts, wts, wts, wts, wts, wts, indexing='ij')

    # Flatten for vectorized evaluation
    Xf = X.ravel()
    Yf = Y.ravel()
    Zf = Z.ravel()
    VXf = VX.ravel()
    VYf = VY.ravel()
    VZf = VZ.ravel()
    WF = (WX * WY * WZ * WVX * WVY * WVZ).ravel()
    # Evaluate P at all points
    Pf = P_X_vectorized(Xf, Yf, Zf, VXf, VYf, VZf, mu)

    # Integral is sum(P * weight) * volume factor
    integral = np.sum(Pf * WF) * (0.5*dx) * (0.5*dy) * (0.5*dz) * (0.5*dvx) * (0.5*dvy) * (0.5*dvz)
    #return integral, y_points
    return integral

In [9]:
import scipy.integrate as integrate
from scipy.special import roots_legendre
from scipy.optimize import minimize_scalar
import warnings

def gauss_kronrod_6d_integral(func, bounds, n_gauss=7, n_kronrod=15, mu=1, adaptive_vz=True, 
                              vz_peak_tolerance=1e-3, max_subdivisions=5):
    """
    Calculate 6-D integral using Gauss-Kronrod quadrature with adaptive handling of sharp peaks in vz.
    
    Parameters:
    -----------
    func : callable
        Function to integrate: f(x, y, z, vx, vy, vz, mu)
    bounds : list of tuples
        Integration bounds for each dimension [(xmin, xmax), (ymin, ymax), ...]
    n_gauss : int
        Number of Gauss points (default 7)
    n_kronrod : int  
        Number of Kronrod points (default 15)
    mu : float
        Gravitational parameter
    adaptive_vz : bool
        Whether to use adaptive integration for vz direction
    vz_peak_tolerance : float
        Tolerance for detecting sharp peaks in vz
    max_subdivisions : int
        Maximum number of subdivisions for adaptive integration
        
    Returns:
    --------
    integral : float
        Estimated integral value
    error_estimate : float
        Error estimate from Gauss-Kronrod difference
    """
    
    # Get Gauss and Kronrod points and weights
    gauss_pts, gauss_wts = roots_legendre(n_gauss)
    kronrod_pts, kronrod_wts = roots_legendre(n_kronrod)
    
    def integrate_1d_gauss_kronrod(f, a, b, gauss_pts, gauss_wts, kronrod_pts, kronrod_wts):
        """1D Gauss-Kronrod integration with error estimation"""
        # Transform points to [a, b]
        gauss_x = 0.5 * (b - a) * gauss_pts + 0.5 * (a + b)
        kronrod_x = 0.5 * (b - a) * kronrod_pts + 0.5 * (a + b)
        
        # Evaluate function
        gauss_vals = np.array([f(x) for x in gauss_x])
        kronrod_vals = np.array([f(x) for x in kronrod_x])
        
        # Calculate integrals
        gauss_integral = 0.5 * (b - a) * np.sum(gauss_wts * gauss_vals)
        kronrod_integral = 0.5 * (b - a) * np.sum(kronrod_wts * kronrod_vals)
        
        # Error estimate
        error_estimate = abs(kronrod_integral - gauss_integral)
        
        return kronrod_integral, error_estimate
    
    def adaptive_vz_integration(f_vz, vz_bounds, tolerance=vz_peak_tolerance, max_depth=max_subdivisions):
        """Adaptive integration for vz direction to handle sharp peaks"""
        
        def integrate_interval(a, b, depth=0):
            if depth >= max_depth:
                return integrate_1d_gauss_kronrod(f_vz, a, b, gauss_pts, gauss_wts, kronrod_pts, kronrod_wts)
            
            # Check for sharp peaks by sampling at multiple points
            vz_test = np.linspace(a, b, 10)
            f_test = np.array([f_vz(vz) for vz in vz_test])
            
            # Detect sharp peaks (large variations)
            f_max = np.max(f_test)
            f_min = np.min(f_test)
            variation = (f_max - f_min) / (f_max + 1e-15)
            
            if variation > tolerance and depth < max_depth:
                # Subdivide interval
                mid = 0.5 * (a + b)
                left_integral, left_error = integrate_interval(a, mid, depth + 1)
                right_integral, right_error = integrate_interval(mid, b, depth + 1)
                return left_integral + right_integral, left_error + right_error
            else:
                return integrate_1d_gauss_kronrod(f_vz, a, b, gauss_pts, gauss_wts, kronrod_pts, kronrod_wts)
        
        return integrate_interval(vz_bounds[0], vz_bounds[1])
    
    # Extract bounds
    x_bounds, y_bounds, z_bounds, vx_bounds, vy_bounds, vz_bounds = bounds
    
    if adaptive_vz:
        # Use adaptive integration for vz direction
        def f_vz(vz):
            def integrand_5d(x, y, z, vx, vy):
                return func(x, y, z, vx, vy, vz, mu)
            
            # Integrate over other 5 dimensions using standard Gauss-Kronrod
            result = integrate_5d_gauss_kronrod(integrand_5d, [x_bounds, y_bounds, z_bounds, vx_bounds, vy_bounds],
                                             gauss_pts, gauss_wts, kronrod_pts, kronrod_wts)
            return result
        
        integral, error_estimate = adaptive_vz_integration(f_vz, vz_bounds)
        
    else:
        # Standard 6D Gauss-Kronrod integration
        integral, error_estimate = integrate_6d_gauss_kronrod(func, bounds, gauss_pts, gauss_wts, 
                                                            kronrod_pts, kronrod_wts, mu)
    
    return integral, error_estimate

def integrate_5d_gauss_kronrod(func, bounds, gauss_pts, gauss_wts, kronrod_pts, kronrod_wts):
    """5D integration using Gauss-Kronrod quadrature"""
    x_bounds, y_bounds, z_bounds, vx_bounds, vy_bounds = bounds
    
    # Create meshgrid for all dimensions
    x_pts = 0.5 * (x_bounds[1] - x_bounds[0]) * kronrod_pts + 0.5 * (x_bounds[1] + x_bounds[0])
    y_pts = 0.5 * (y_bounds[1] - y_bounds[0]) * kronrod_pts + 0.5 * (y_bounds[1] + y_bounds[0])
    z_pts = 0.5 * (z_bounds[1] - z_bounds[0]) * kronrod_pts + 0.5 * (z_bounds[1] + z_bounds[0])
    vx_pts = 0.5 * (vx_bounds[1] - vx_bounds[0]) * kronrod_pts + 0.5 * (vx_bounds[1] + vx_bounds[0])
    vy_pts = 0.5 * (vy_bounds[1] - vy_bounds[0]) * kronrod_pts + 0.5 * (vy_bounds[1] + vy_bounds[0])
    
    X, Y, Z, VX, VY = np.meshgrid(x_pts, y_pts, z_pts, vx_pts, vy_pts, indexing='ij')
    WX, WY, WZ, WVX, WVY = np.meshgrid(kronrod_wts, kronrod_wts, kronrod_wts, kronrod_wts, kronrod_wts, indexing='ij')
    
    # Flatten for evaluation
    Xf, Yf, Zf, VXf, VYf = X.ravel(), Y.ravel(), Z.ravel(), VX.ravel(), VY.ravel()
    Wf = (WX * WY * WZ * WVX * WVY).ravel()
    
    # Evaluate function
    f_vals = np.array([func(x, y, z, vx, vy) for x, y, z, vx, vy in zip(Xf, Yf, Zf, VXf, VYf)])
    
    # Calculate integral
    volume_factor = (0.5 * (x_bounds[1] - x_bounds[0]) * 
                    0.5 * (y_bounds[1] - y_bounds[0]) * 
                    0.5 * (z_bounds[1] - z_bounds[0]) * 
                    0.5 * (vx_bounds[1] - vx_bounds[0]) * 
                    0.5 * (vy_bounds[1] - vy_bounds[0]))
    
    integral = np.sum(f_vals * Wf) * volume_factor
    return integral

def integrate_6d_gauss_kronrod(func, bounds, gauss_pts, gauss_wts, kronrod_pts, kronrod_wts, mu):
    """6D integration using Gauss-Kronrod quadrature"""
    x_bounds, y_bounds, z_bounds, vx_bounds, vy_bounds, vz_bounds = bounds
    
    # Create meshgrid for all dimensions
    x_pts = 0.5 * (x_bounds[1] - x_bounds[0]) * kronrod_pts + 0.5 * (x_bounds[1] + x_bounds[0])
    y_pts = 0.5 * (y_bounds[1] - y_bounds[0]) * kronrod_pts + 0.5 * (y_bounds[1] + y_bounds[0])
    z_pts = 0.5 * (z_bounds[1] - z_bounds[0]) * kronrod_pts + 0.5 * (z_bounds[1] + z_bounds[0])
    vx_pts = 0.5 * (vx_bounds[1] - vx_bounds[0]) * kronrod_pts + 0.5 * (vx_bounds[1] + vx_bounds[0])
    vy_pts = 0.5 * (vy_bounds[1] - vy_bounds[0]) * kronrod_pts + 0.5 * (vy_bounds[1] + vy_bounds[0])
    vz_pts = 0.5 * (vz_bounds[1] - vz_bounds[0]) * kronrod_pts + 0.5 * (vz_bounds[1] + vz_bounds[0])
    
    X, Y, Z, VX, VY, VZ = np.meshgrid(x_pts, y_pts, z_pts, vx_pts, vy_pts, vz_pts, indexing='ij')
    WX, WY, WZ, WVX, WVY, WVZ = np.meshgrid(kronrod_wts, kronrod_wts, kronrod_wts, kronrod_wts, kronrod_wts, kronrod_wts, indexing='ij')
    
    # Flatten for evaluation
    Xf, Yf, Zf, VXf, VYf, VZf = X.ravel(), Y.ravel(), Z.ravel(), VX.ravel(), VY.ravel(), VZ.ravel()
    Wf = (WX * WY * WZ * WVX * WVY * WVZ).ravel()
    
    # Evaluate function
    f_vals = np.array([func(x, y, z, vx, vy, vz, mu) for x, y, z, vx, vy, vz in zip(Xf, Yf, Zf, VXf, VYf, VZf)])
    
    # Calculate integral
    volume_factor = (0.5 * (x_bounds[1] - x_bounds[0]) * 
                    0.5 * (y_bounds[1] - y_bounds[0]) * 
                    0.5 * (z_bounds[1] - z_bounds[0]) * 
                    0.5 * (vx_bounds[1] - vx_bounds[0]) * 
                    0.5 * (vy_bounds[1] - vy_bounds[0]) * 
                    0.5 * (vz_bounds[1] - vz_bounds[0]))
    
    integral = np.sum(f_vals * Wf) * volume_factor
    return integral, 0.0  # No error estimate for full 6D case


In [10]:
def gauss_kronrod_6d_vectorized(func, bounds, n_gauss=7, n_kronrod=15, mu=1, 
                                adaptive_vz=True, vz_peak_tolerance=1e-3, max_subdivisions=5):
    """
    Optimized vectorized version of 6-D Gauss-Kronrod integration.
    Uses vectorized evaluation where possible for better performance.
    """
    
    # Get Gauss and Kronrod points and weights
    gauss_pts, gauss_wts = roots_legendre(n_gauss)
    kronrod_pts, kronrod_wts = roots_legendre(n_kronrod)
    
    def integrate_1d_gauss_kronrod_vectorized(f, a, b, gauss_pts, gauss_wts, kronrod_pts, kronrod_wts):
        """1D Gauss-Kronrod integration with vectorized evaluation"""
        # Transform points to [a, b]
        gauss_x = 0.5 * (b - a) * gauss_pts + 0.5 * (a + b)
        kronrod_x = 0.5 * (b - a) * kronrod_pts + 0.5 * (a + b)
        
        # Evaluate function (vectorized if possible)
        try:
            gauss_vals = f(gauss_x)
            kronrod_vals = f(kronrod_x)
        except:
            # Fallback to element-wise evaluation
            gauss_vals = np.array([f(x) for x in gauss_x])
            kronrod_vals = np.array([f(x) for x in kronrod_x])
        
        # Calculate integrals
        gauss_integral = 0.5 * (b - a) * np.sum(gauss_wts * gauss_vals)
        kronrod_integral = 0.5 * (b - a) * np.sum(kronrod_wts * kronrod_vals)
        
        # Error estimate
        error_estimate = abs(kronrod_integral - gauss_integral)
        
        return kronrod_integral, error_estimate
    
    def adaptive_vz_integration_vectorized(f_vz, vz_bounds, tolerance=vz_peak_tolerance, max_depth=max_subdivisions):
        """Adaptive integration for vz direction with vectorized evaluation"""
        
        def integrate_interval(a, b, depth=0):
            if depth >= max_depth:
                return integrate_1d_gauss_kronrod_vectorized(f_vz, a, b, gauss_pts, gauss_wts, kronrod_pts, kronrod_wts)
            
            # Check for sharp peaks by sampling at multiple points
            vz_test = np.linspace(a, b, 10)
            try:
                f_test = f_vz(vz_test)  # Try vectorized evaluation
            except:
                f_test = np.array([f_vz(vz) for vz in vz_test])  # Fallback to element-wise
            
            # Detect sharp peaks (large variations)
            f_max = np.max(f_test)
            f_min = np.min(f_test)
            variation = (f_max - f_min) / (f_max + 1e-15)
            
            if variation > tolerance and depth < max_depth:
                # Subdivide interval
                mid = 0.5 * (a + b)
                left_integral, left_error = integrate_interval(a, mid, depth + 1)
                right_integral, right_error = integrate_interval(mid, b, depth + 1)
                return left_integral + right_integral, left_error + right_error
            else:
                return integrate_1d_gauss_kronrod_vectorized(f_vz, a, b, gauss_pts, gauss_wts, kronrod_pts, kronrod_wts)
        
        return integrate_interval(vz_bounds[0], vz_bounds[1])
    
    # Extract bounds
    x_bounds, y_bounds, z_bounds, vx_bounds, vy_bounds, vz_bounds = bounds
    
    if adaptive_vz:
        # Use adaptive integration for vz direction with vectorized 5D integration
        def f_vz_vectorized(vz):
            def integrand_5d_vectorized(x, y, z, vx, vy):
                return func(x, y, z, vx, vy, vz, mu)
            
            # Integrate over other 5 dimensions using vectorized Gauss-Kronrod
            result = integrate_5d_gauss_kronrod_vectorized(integrand_5d_vectorized, 
                                                        [x_bounds, y_bounds, z_bounds, vx_bounds, vy_bounds],
                                                        gauss_pts, gauss_wts, kronrod_pts, kronrod_wts)
            return result
        
        integral, error_estimate = adaptive_vz_integration_vectorized(f_vz_vectorized, vz_bounds)
        
    else:
        # Standard 6D Gauss-Kronrod integration with vectorization
        integral, error_estimate = integrate_6d_gauss_kronrod_vectorized(func, bounds, gauss_pts, gauss_wts, 
                                                                       kronrod_pts, kronrod_wts, mu)
    
    return integral, error_estimate

def integrate_5d_gauss_kronrod_vectorized(func, bounds, gauss_pts, gauss_wts, kronrod_pts, kronrod_wts):
    """5D integration using vectorized Gauss-Kronrod quadrature"""
    x_bounds, y_bounds, z_bounds, vx_bounds, vy_bounds = bounds
    
    # Create meshgrid for all dimensions
    x_pts = 0.5 * (x_bounds[1] - x_bounds[0]) * kronrod_pts + 0.5 * (x_bounds[1] + x_bounds[0])
    y_pts = 0.5 * (y_bounds[1] - y_bounds[0]) * kronrod_pts + 0.5 * (y_bounds[1] + y_bounds[0])
    z_pts = 0.5 * (z_bounds[1] - z_bounds[0]) * kronrod_pts + 0.5 * (z_bounds[1] + z_bounds[0])
    vx_pts = 0.5 * (vx_bounds[1] - vx_bounds[0]) * kronrod_pts + 0.5 * (vx_bounds[1] + vx_bounds[0])
    vy_pts = 0.5 * (vy_bounds[1] - vy_bounds[0]) * kronrod_pts + 0.5 * (vy_bounds[1] + vy_bounds[0])
    
    X, Y, Z, VX, VY = np.meshgrid(x_pts, y_pts, z_pts, vx_pts, vy_pts, indexing='ij')
    WX, WY, WZ, WVX, WVY = np.meshgrid(kronrod_wts, kronrod_wts, kronrod_wts, kronrod_wts, kronrod_wts, indexing='ij')
    
    # Flatten for evaluation
    Xf, Yf, Zf, VXf, VYf = X.ravel(), Y.ravel(), Z.ravel(), VX.ravel(), VY.ravel()
    Wf = (WX * WY * WZ * WVX * WVY).ravel()
    
    # Try vectorized evaluation first
    try:
        f_vals = func(Xf, Yf, Zf, VXf, VYf)
    except:
        # Fallback to element-wise evaluation
        f_vals = np.array([func(x, y, z, vx, vy) for x, y, z, vx, vy in zip(Xf, Yf, Zf, VXf, VYf)])
    
    # Calculate integral
    volume_factor = (0.5 * (x_bounds[1] - x_bounds[0]) * 
                    0.5 * (y_bounds[1] - y_bounds[0]) * 
                    0.5 * (z_bounds[1] - z_bounds[0]) * 
                    0.5 * (vx_bounds[1] - vx_bounds[0]) * 
                    0.5 * (vy_bounds[1] - vy_bounds[0]))
    
    integral = np.sum(f_vals * Wf) * volume_factor
    return integral

def integrate_6d_gauss_kronrod_vectorized(func, bounds, gauss_pts, gauss_wts, kronrod_pts, kronrod_wts, mu):
    """6D integration using vectorized Gauss-Kronrod quadrature"""
    x_bounds, y_bounds, z_bounds, vx_bounds, vy_bounds, vz_bounds = bounds
    
    # Create meshgrid for all dimensions
    x_pts = 0.5 * (x_bounds[1] - x_bounds[0]) * kronrod_pts + 0.5 * (x_bounds[1] + x_bounds[0])
    y_pts = 0.5 * (y_bounds[1] - y_bounds[0]) * kronrod_pts + 0.5 * (y_bounds[1] + y_bounds[0])
    z_pts = 0.5 * (z_bounds[1] - z_bounds[0]) * kronrod_pts + 0.5 * (z_bounds[1] + z_bounds[0])
    vx_pts = 0.5 * (vx_bounds[1] - vx_bounds[0]) * kronrod_pts + 0.5 * (vx_bounds[1] + vx_bounds[0])
    vy_pts = 0.5 * (vy_bounds[1] - vy_bounds[0]) * kronrod_pts + 0.5 * (vy_bounds[1] + vy_bounds[0])
    vz_pts = 0.5 * (vz_bounds[1] - vz_bounds[0]) * kronrod_pts + 0.5 * (vz_bounds[1] + vz_bounds[0])
    
    X, Y, Z, VX, VY, VZ = np.meshgrid(x_pts, y_pts, z_pts, vx_pts, vy_pts, vz_pts, indexing='ij')
    WX, WY, WZ, WVX, WVY, WVZ = np.meshgrid(kronrod_wts, kronrod_wts, kronrod_wts, kronrod_wts, kronrod_wts, kronrod_wts, indexing='ij')
    
    # Flatten for evaluation
    Xf, Yf, Zf, VXf, VYf, VZf = X.ravel(), Y.ravel(), Z.ravel(), VX.ravel(), VY.ravel(), VZ.ravel()
    Wf = (WX * WY * WZ * WVX * WVY * WVZ).ravel()
    
    # Try vectorized evaluation first
    try:
        f_vals = func(Xf, Yf, Zf, VXf, VYf, VZf, mu)
    except:
        # Fallback to element-wise evaluation
        f_vals = np.array([func(x, y, z, vx, vy, vz, mu) for x, y, z, vx, vy, vz in zip(Xf, Yf, Zf, VXf, VYf, VZf)])
    
    # Calculate integral
    volume_factor = (0.5 * (x_bounds[1] - x_bounds[0]) * 
                    0.5 * (y_bounds[1] - y_bounds[0]) * 
                    0.5 * (z_bounds[1] - z_bounds[0]) * 
                    0.5 * (vx_bounds[1] - vx_bounds[0]) * 
                    0.5 * (vy_bounds[1] - vy_bounds[0]) * 
                    0.5 * (vz_bounds[1] - vz_bounds[0]))
    
    integral = np.sum(f_vals * Wf) * volume_factor
    return integral, 0.0  # No error estimate for full 6D case


In [ ]:
import time
xmin, xmax = (0.9039710143502464, 1.0960289856497536)
ymin, ymax = (-0.09602898564975362, 0.09602898564975362)
zmin, zmax = (-0.09602898564975362, 0.09602898564975362)
vxmin, vxmax = (-0.5064245183055923, 0.5064245183055923)
vymin, vymax = (5.777598212496408, 6.790447249107593)
vzmin, vzmax = (-0.5064245183055923, 0.5064245183055923)

# Test the Gauss-Kronrod implementation
print("Testing Gauss-Kronrod 6D integration with adaptive vz handling...")
print("=" * 60)

# Define bounds (same as before)
bounds = [(xmin, xmax), (ymin, ymax), (zmin, zmax), (vxmin, vxmax), (vymin, vymax), (vzmin, vzmax)]

# Test 1: Standard Gauss-Kronrod (non-adaptive)
print("1. Standard Gauss-Kronrod integration:")
start_time = time.time()
integral_std, error_std = gauss_kronrod_6d_integral(P_X, bounds, mu=mu, adaptive_vz=False, n_gauss=5, n_kronrod=11)
time_std = time.time() - start_time
print(f"   Integral: {integral_std:.6e}")
print(f"   Error estimate: {error_std:.6e}")
print(f"   Time: {time_std:.2f} seconds")
print()

# Test 2: Adaptive Gauss-Kronrod for vz direction
print("2. Adaptive Gauss-Kronrod integration (vz adaptive):")
start_time = time.time()
integral_adaptive, error_adaptive = gauss_kronrod_6d_integral(P_X, bounds, mu=mu, adaptive_vz=True, 
                                                             vz_peak_tolerance=1e-2, max_subdivisions=3)
time_adaptive = time.time() - start_time
print(f"   Integral: {integral_adaptive:.6e}")
print(f"   Error estimate: {error_adaptive:.6e}")
print(f"   Time: {time_adaptive:.2f} seconds")
print()

# Test 3: Vectorized version
print("3. Vectorized Gauss-Kronrod integration:")
start_time = time.time()
integral_vec, error_vec = gauss_kronrod_6d_vectorized(P_X_vectorized, bounds, mu=mu, adaptive_vz=True,
                                                     vz_peak_tolerance=1e-2, max_subdivisions=3)
time_vec = time.time() - start_time
print(f"   Integral: {integral_vec:.6e}")
print(f"   Error estimate: {error_vec:.6e}")
print(f"   Time: {time_vec:.2f} seconds")
print()

# Compare with previous results
print("Comparison with previous methods:")
print(f"   Monte Carlo (N=1e7): {I:.6e} ± {dI:.2e}")
print(f"   Importance sampling: {I:.6e} ± {dI:.2e}")
print(f"   Standard Gauss-Kronrod: {integral_std:.6e}")
print(f"   Adaptive Gauss-Kronrod: {integral_adaptive:.6e}")
print(f"   Vectorized Gauss-Kronrod: {integral_vec:.6e}")
print()

# Performance comparison
print("Performance comparison:")
print(f"   Standard GK: {time_std:.2f}s")
print(f"   Adaptive GK: {time_adaptive:.2f}s")
print(f"   Vectorized GK: {time_vec:.2f}s")
print(f"   Speedup (vectorized vs standard): {time_std/time_vec:.2f}x")


Testing Gauss-Kronrod 6D integration with adaptive vz handling...
1. Standard Gauss-Kronrod integration:


C:\Users\aguju\AppData\Local\Temp\ipykernel_15728\3075411317.py:7: RuntimeWarning: divide by zero encountered in scalar divide
  det = 1.0/np.linalg.det(J)


   Integral: inf
   Error estimate: 0.000000e+00
   Time: 320.81 seconds

2. Adaptive Gauss-Kronrod integration (vz adaptive):


C:\Users\aguju\AppData\Local\Temp\ipykernel_15728\4212434350.py:75: RuntimeWarning: invalid value encountered in scalar divide
  variation = (f_max - f_min) / (f_max + 1e-15)


   Integral: 6.755694e-05
   Error estimate: 5.387660e-05
   Time: 9900.42 seconds

3. Vectorized Gauss-Kronrod integration:


In [ ]:
def advanced_gauss_kronrod_6d(func, bounds, mu=1, peak_detection_threshold=1e-2, 
                             max_adaptive_depth=5, convergence_tolerance=1e-6):
    """
    Advanced Gauss-Kronrod integration with sophisticated peak detection and adaptive refinement.
    
    Features:
    - Automatic peak detection in vz direction
    - Adaptive mesh refinement
    - Convergence checking
    - Error estimation
    """
    
    def detect_peaks_1d(f, bounds, n_samples=50):
        """Detect sharp peaks in 1D function"""
        x = np.linspace(bounds[0], bounds[1], n_samples)
        y = np.array([f(xi) for xi in x])
        
        # Find local maxima
        peaks = []
        for i in range(1, len(y)-1):
            if y[i] > y[i-1] and y[i] > y[i+1]:
                peaks.append(x[i])
        
        # Calculate peak sharpness
        if len(peaks) > 0:
            peak_values = [f(p) for p in peaks]
            max_peak = max(peak_values)
            avg_peak = np.mean(peak_values)
            sharpness = max_peak / (avg_peak + 1e-15)
            return peaks, sharpness
        else:
            return [], 1.0
    
    def adaptive_integration_with_peaks(f, bounds, depth=0, max_depth=max_adaptive_depth):
        """Adaptive integration with peak-aware subdivision"""
        
        if depth >= max_depth:
            # Use standard Gauss-Kronrod
            gauss_pts, gauss_wts = roots_legendre(7)
            kronrod_pts, kronrod_wts = roots_legendre(15)
            
            a, b = bounds
            gauss_x = 0.5 * (b - a) * gauss_pts + 0.5 * (a + b)
            kronrod_x = 0.5 * (b - a) * kronrod_pts + 0.5 * (a + b)
            
            gauss_vals = np.array([f(x) for x in gauss_x])
            kronrod_vals = np.array([f(x) for x in kronrod_x])
            
            gauss_integral = 0.5 * (b - a) * np.sum(gauss_wts * gauss_vals)
            kronrod_integral = 0.5 * (b - a) * np.sum(kronrod_wts * kronrod_vals)
            error_estimate = abs(kronrod_integral - gauss_integral)
            
            return kronrod_integral, error_estimate
        
        # Detect peaks
        peaks, sharpness = detect_peaks_1d(f, bounds)
        
        if sharpness > peak_detection_threshold and len(peaks) > 0:
            # Subdivide around peaks
            a, b = bounds
            subdivisions = [a]
            
            for peak in peaks:
                # Add points around peak
                peak_width = (b - a) * 0.1  # 10% of interval width
                subdivisions.extend([peak - peak_width, peak, peak + peak_width])
            
            subdivisions.append(b)
            subdivisions = sorted(list(set(subdivisions)))
            subdivisions = [x for x in subdivisions if a <= x <= b]
            
            # Integrate each subinterval
            total_integral = 0.0
            total_error = 0.0
            
            for i in range(len(subdivisions) - 1):
                sub_bounds = (subdivisions[i], subdivisions[i+1])
                sub_integral, sub_error = adaptive_integration_with_peaks(f, sub_bounds, depth + 1, max_depth)
                total_integral += sub_integral
                total_error += sub_error
            
            return total_integral, total_error
        else:
            # No significant peaks, use standard integration
            return adaptive_integration_with_peaks(f, bounds, max_depth, max_depth)
    
    # Extract bounds
    x_bounds, y_bounds, z_bounds, vx_bounds, vy_bounds, vz_bounds = bounds
    
    # Create function for vz integration
    def f_vz_advanced(vz):
        def integrand_5d(x, y, z, vx, vy):
            return func(x, y, z, vx, vy, vz, mu)
        
        # Use standard 5D integration for other dimensions
        return integrate_5d_gauss_kronrod_vectorized(integrand_5d, 
                                                   [x_bounds, y_bounds, z_bounds, vx_bounds, vy_bounds],
                                                   *roots_legendre(7), *roots_legendre(15))
    
    # Apply adaptive integration to vz direction
    integral, error_estimate = adaptive_integration_with_peaks(f_vz_advanced, vz_bounds)
    
    return integral, error_estimate

# Test the advanced implementation
print("Testing Advanced Gauss-Kronrod with peak detection...")
print("=" * 60)

start_time = time.time()
integral_advanced, error_advanced = advanced_gauss_kronrod_6d(P_X, bounds, mu=mu)
time_advanced = time.time() - start_time

print(f"Advanced Gauss-Kronrod result:")
print(f"   Integral: {integral_advanced:.6e}")
print(f"   Error estimate: {error_advanced:.6e}")
print(f"   Time: {time_advanced:.2f} seconds")
print()

print("Final comparison of all methods:")
print(f"   Monte Carlo:           {I:.6e} ± {dI:.2e}")
print(f"   Standard GK:           {integral_std:.6e}")
print(f"   Adaptive GK:           {integral_adaptive:.6e}")
print(f"   Vectorized GK:         {integral_vec:.6e}")
print(f"   Advanced GK:           {integral_advanced:.6e}")
print()

print("Performance summary:")
print(f"   Standard GK:     {time_std:.2f}s")
print(f"   Adaptive GK:     {time_adaptive:.2f}s") 
print(f"   Vectorized GK:   {time_vec:.2f}s")
print(f"   Advanced GK:     {time_advanced:.2f}s")


In [ ]:
from Utils import trasformation_X_to_E, jacobian_XoE, P_E

def P_X(x: float, y: float, z: float, vx: float, vy: float, vz: float) -> float:
    a, e, i, Omega, w, M = trasformation_X_to_E(x, y, z, vx, vy, vz, mu)
    J = jacobian_XoE(a,e,i,Omega,w,M,mu)
    #det = np.linalg.det(J)
    det = 1.0/np.linalg.det(J)
    P = P_E() * abs(det)
    return P
    
def P_X_vectorized(x: np.array, y: np.array, z: np.array, vx: np.array, vy: np.array, vz: np.array, mu: float) -> np.array:
    """
    Vectorized version: x, y, vx, vy are arrays (or scalars).
    Returns array of P values.
    """
    x = np.asarray(x)
    y = np.asarray(y)
    z = np.asarray(z)
    vx = np.asarray(vx)
    vy = np.asarray(vy)
    vz = np.asarray(vz)
    # Prepare output array
    shape = np.broadcast(x, y, z, vx, vy, vz).shape
    P = np.empty(shape, dtype=float)

    # Flatten for iteration if needed
    x_flat = x.ravel()
    y_flat = y.ravel()
    z_flat = z.ravel()
    vx_flat = vx.ravel()
    vy_flat = vy.ravel()
    vz_flat = vz.ravel()

    for idx in range(x_flat.size):
        a, e, i, Omega, w, M = trasformation_X_to_E(x_flat[idx], y_flat[idx], z_flat[idx], vx_flat[idx], vy_flat[idx], vz_flat[idx], mu)
        J = jacobian_XoE(a,e,i,Omega,w,M,mu)
        det = np.linalg.det(J)
        inv_det = 1.0/det
        P.flat[idx] = P_E() * abs(inv_det)

    return P.reshape(shape)

def surface_integral_P_X(center, widths, n_points=8, mu=1):
    """
    Calculate the surface integral of P_xyvxvy in a hypercube centered at (x, y, vx, vy)
    with dimensions (dx, dy, dvx, dvy) using Gauss-Legendre quadrature.

    Parameters:
        center: tuple/list/array of (x, y, vx, vy) center
        widths: tuple/list/array of (dx, dy, dvx, dvy) side lengths
        n_points: number of quadrature points per dimension

    Returns:
        Integral (float)
    """
    from numpy.polynomial.legendre import leggauss

    x0, y0, z0, vx0, vy0, vz0 = center
    dx, dy, dz, dvx, dvy, dvz = widths

    # Get Gauss-Legendre points and weights for [-1, 1]
    pts, wts = leggauss(n_points)

    # Map points from [-1, 1] to [center-width/2, center+width/2] for each dimension
    x_pts = x0 + 0.5*dx*pts
    y_pts = y0 + 0.5*dy*pts
    z_pts = z0 + 0.5*dz*pts
    vx_pts = vx0 + 0.5*dvx*pts
    vy_pts = vy0 + 0.5*dvy*pts
    vz_pts = vz0 + 0.5*dvz*pts

    # Create meshgrid of all quadrature points
    X, Y, Z, VX, VY, VZ = np.meshgrid(x_pts, y_pts, z_pts, vx_pts, vy_pts, vz_pts, indexing='ij')
    WX, WY, WZ, WVX, WVY, WVZ = np.meshgrid(wts, wts, wts, wts, wts, wts, indexing='ij')

    # Flatten for vectorized evaluation
    Xf = X.ravel()
    Yf = Y.ravel()
    Zf = Z.ravel()
    VXf = VX.ravel()
    VYf = VY.ravel()
    VZf = VZ.ravel()
    WF = (WX * WY * WZ * WVX * WVY * WVZ).ravel()
    # Evaluate P at all points
    Pf = P_X_vectorized(Xf, Yf, Zf, VXf, VYf, VZf, mu)

    # Integral is sum(P * weight) * volume factor
    integral = np.sum(Pf * WF) * (0.5*dx) * (0.5*dy) * (0.5*dz) * (0.5*dvx) * (0.5*dvy) * (0.5*dvz)
    #return integral, y_points
    return integral